# 02 · Analysis tools and writing a bot

Building on `01_game_basics.ipynb`. This notebook covers the analysis primitives the framework gives you, then walks through writing a real bot end to end and pitting it against the included opponents.

Estimated time: 20 minutes.

In [ ]:
!pip install --quiet hexbot

## Threats and winning moves

A "threat" is a move that creates an immediate winning option (i.e. that side could place a sixth in-a-row stone next turn unless blocked). `find_threats` returns every such cell for either player; `find_winning_moves` returns moves that complete a line right now.

In [ ]:
from hexbot import HexGame, find_threats, find_winning_moves, find_forced_move

g = HexGame()
# Set up a five-in-a-row for P0 along q axis
g.place(0, 0)
g.place(0, 5); g.place(0, 6)
g.place(1, 0); g.place(2, 0)
g.place(1, 5); g.place(1, 6)
g.place(3, 0); g.place(4, 0)
g.place(2, 5); g.place(2, 6)
# now P0 to play with 5 stones at (0,0)..(4,0); (5,0) or (-1,0) completes 6

print(f"P0 winning moves: {find_winning_moves(g, player=0)}")
print(f"P1 winning moves: {find_winning_moves(g, player=1)}")
print(f"forced move:      {find_forced_move(g)}  (a smart bot should play this)")

## Move scoring

`evaluate_moves(game, n)` ranks the top `n` candidate cells using the C heuristic (which scores line extension potential, blocking value, and proximity to existing stones). It is the cheapest way to get a useful shortlist.

In [ ]:
from hexbot import evaluate_moves

g = HexGame()
g.place(0, 0)
g.place(1, 0); g.place(1, -1)

for (move, score) in evaluate_moves(g, top_n=5):
    print(f"  {move}  score={score}")

## Alpha-beta search

For tactical correctness (not falling for sneaky two-move losses), the C engine ships an alpha-beta search with transposition tables, killer heuristics, and late move reduction.

Depth is in **half-moves** (plies). Depth 4 = both players make two stones each. Depth 8 covers four turns.

In [ ]:
g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)

result = g.search(depth=6)
print(f"best move:  {result['best_move']}")
print(f"evaluation: {result['value']:+.2f}   (positive = good for side to move)")
print(f"nodes:      {result['nodes']:,}")

## Endgame solver

When the game gets close to a forced result, `solve()` brute-forces the outcome (subject to a time and depth limit). Returns `'win' / 'loss' / 'draw' / 'unknown'`.

In [ ]:
from hexbot import solve

g = HexGame()
g.place(0, 0)
g.place(0, 5); g.place(0, 6)
g.place(1, 0); g.place(2, 0)
g.place(1, 5); g.place(1, 6)
g.place(3, 0); g.place(4, 0)
g.place(2, 5); g.place(2, 6)
# P0 to play, five-in-a-row already on the board

out = solve(g, max_depth=6, time_limit=2.0)
print(f"result: {out['result']}")
print(f"move:   {out['move']}")

## Writing a bot from scratch

Anything callable as `f(game) -> (q, r)` is a bot. We will start with the simplest possible bot, then make it smarter step by step.

### Bot 1: pick the top heuristic move

In [ ]:
from hexbot import HexGame, evaluate_moves

def top_heuristic_bot(game):
    return evaluate_moves(game, top_n=1)[0][0]

# Try it on a fresh game
g = HexGame(); g.place(0, 0)
print(f"first response: {top_heuristic_bot(g)}")

### Bot 2: always answer forced moves first

A bot that ignores forced moves loses the moment the opponent gets a five-in-a-row. Adding `find_forced_move` is the single biggest jump in strength you can make on top of a heuristic bot.

In [ ]:
from hexbot import find_forced_move, evaluate_moves

def safer_bot(game):
    forced = find_forced_move(game)
    if forced is not None:
        return forced
    return evaluate_moves(game, top_n=1)[0][0]

### Bot 3: alpha-beta with the forced-move shortcut

Now we look several turns ahead with the C engine's search.

In [ ]:
def deep_bot(game, depth=6):
    forced = find_forced_move(game)
    if forced is not None:
        return forced
    return game.search(depth=depth)['best_move']

## Pit them against each other (Arena)

`Arena(bot_a, bot_b, num_games=N).play()` runs N games (split evenly between the two playing P0/P1) and returns win rates.

In [ ]:
from hexbot import Bot, Arena

result = Arena(deep_bot, top_heuristic_bot, num_games=6).play(verbose=False)
print(f"deep_bot vs top_heuristic_bot: {result}")

result = Arena(deep_bot, Bot.heuristic(), num_games=4).play(verbose=False)
print(f"deep_bot vs Bot.heuristic():   {result}")

result = Arena(deep_bot, Bot.random(), num_games=4).play(verbose=False)
print(f"deep_bot vs Bot.random():      {result}")

## Registering your bot in the plugin system

If you intend to reuse `deep_bot` across scripts and dashboards, register it. After this, anything in the framework that accepts a bot name (`--opponent deep`, dashboard dropdown, leaderboard) finds it by string.

In [ ]:
from hexbot import BotProtocol, register_bot, registered_bots

class DeepBot(BotProtocol):
    def __init__(self, depth=6):
        self.depth = depth
    def best_move(self, game):
        return deep_bot(game, depth=self.depth)

register_bot('deep', DeepBot)
print('registered bots:', sorted(registered_bots().keys()))

## Try it yourself

Extend `deep_bot` so that, in addition to forced-move handling and deep search, it also runs the endgame solver when the total stone count is high. The solver is cheap when the tree is small and can prove a forced win that alpha-beta would miss at the search horizon.

In [ ]:
from hexbot import solve

def hybrid_bot(game, depth=6):
    # TODO:
    # 1. find_forced_move shortcut (already done)
    # 2. if game.total_stones > 80, try solve(game, max_depth=10, time_limit=1.0)
    #    and play the result['move'] if 'result' is 'win'
    # 3. otherwise alpha-beta at given depth
    forced = find_forced_move(game)
    if forced is not None:
        return forced
    return game.search(depth=depth)['best_move']

## Next

You can now build search-based bots. The next notebook shows how to train an AlphaZero-style bot from self-play and how to inspect the training run.

→ [03 · Training Orca (AlphaZero-style)](03_training_orca.ipynb)